## KMeans clustering of taxi pickup and dropoff locations.

In [1]:
import os

import geopandas as gp
import pandas as pd
import pyspark.sql.functions as F
import xyzservices.providers as xyz
from pyspark.ml import Pipeline
from pyspark.ml.clustering import KMeans
from pyspark.ml.feature import VectorAssembler

import geofunctions as S

In [2]:
schema = ",".join(
    [
        "id string",
        "vendor_id string",
        "pickup_datetime timestamp",
        "dropoff_datetime timestamp",
        "passenger_count integer",
        "pickup_longitude double",
        "pickup_latitude double",
        "dropoff_longitude double",
        "dropoff_latitude double",
        "store_and_fwd_flag string",
        "trip_duration integer",  # seconds
    ]
)

In [3]:
xmin, ymin, xmax, ymax = [
    -74.04226680414989,
    40.6059735081628,
    -73.5061146385722,
    40.91340235669589,
]

where = " and ".join(
    [
        f"p_lon between {xmin}D and {xmax}D",
        f"p_lat between {ymin}D and {ymax}D",
        f"d_lon between {xmin}D and {xmax}D",
        f"d_lat between {ymin}D and {ymax}D",
        "duration between 60 and 5400",
        "haversine between 10D and 32000D",
    ]
)

In [4]:
csv_path = os.path.expanduser("~/JHUBigDataClass/data/nyc-taxi-trip-duration/train.csv")


df = (
    spark.read.csv(csv_path, schema=schema, sep=",", mode="DROPMALFORMED")
    .withColumnRenamed("pickup_datetime", "p_date")
    .withColumnRenamed("pickup_longitude", "p_lon")
    .withColumnRenamed("pickup_latitude", "p_lat")
    .withColumnRenamed("dropoff_datetime", "d_date")
    .withColumnRenamed("dropoff_longitude", "d_lon")
    .withColumnRenamed("dropoff_latitude", "d_lat")
    .withColumnRenamed("trip_duration", "duration")
    .withColumn("haversine", S.st_haversine("p_lon", "p_lat", "d_lon", "d_lat"))
    .where(where)
)

### Create a dataframe of all pickups and dropoffs as single xy dataframe.

In [5]:
df1 = df.select(S.st_lontox("p_lon").alias("x"), S.st_lattoy("p_lat").alias("y"))
df2 = df.select(S.st_lontox("d_lon").alias("x"), S.st_lattoy("d_lat").alias("y"))

xy = df1.unionAll(df2).cache()

In [6]:
# xy.show(n=3)

### Create weighted QR cells.

In [7]:
cell = 100.0  # meters
dist = cell * 0.5

xyw = (
    xy.select(S.st_xtoq("x", cell), S.st_ytor("y", cell))
    .groupBy("q", "r")
    .count()
    .select(S.st_qtox("q", cell, dist), S.st_rtoy("r", cell, dist), F.col("count").alias("weight"))
    .cache()
)

### Create ML pipeline

In [8]:
assembler = VectorAssembler(
    inputCols=["x", "y"],
    outputCol="features",
)

kmeans = KMeans(
    predictionCol="clusterId", k=100, maxIter=30, seed=42, tol=0.1, weightCol="weight"
)

pipeline = Pipeline(stages=[assembler, kmeans])

### Create fitted model.

In [9]:
model = pipeline.fit(xyw)

23/08/02 13:04:29 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


### Get fitted KMeans model.

In [10]:
kmeans_model = model.stages[-1]

### Create a pandas dataframe of the cluster centers.

In [11]:
pdf = pd.DataFrame(kmeans_model.clusterCenters(), columns=["x", "y"])
pdf["clusterId"] = pdf.index

### Assign to each qr a cluster id.

In [12]:
pred = model.transform(xyw).drop("features")
pred.show(n=5)

+----------+---------+------+---------+
|         x|        y|weight|clusterId|
+----------+---------+------+---------+
|-8235850.0|4975550.0|   933|       14|
|-8236250.0|4977050.0|  1980|       66|
|-8235550.0|4979750.0|  1274|        6|
|-8237950.0|4972350.0|   996|       87|
|-8238250.0|4970850.0|   511|       87|
+----------+---------+------+---------+
only showing top 5 rows



### Display the cluster centers.

In [13]:
gdf = gp.GeoDataFrame(pdf["clusterId"], geometry=gp.points_from_xy(pdf.x, pdf.y), crs="EPSG:3857")
_ = gdf.sindex # Enable spatial index.

In [14]:
gdf.explore(tiles=xyz.Esri.WorldGrayCanvas)